# TP1 · Softmax regression on 20 Newsgroups

The aim is to classify forum posts into 20 topics and establish a baseline for the later MLP assignments. The workflow covers text preprocessing, Optuna search, training dynamics, convergence timing and final test evaluation.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "tng").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import mlflow
import nltk
import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter

from tng.data import make_loader, preprocess_text, vectorize_data
from tng.training import SoftmaxRegression, evaluate_classifier, train_classifier

SEED = 42
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUNS = ROOT / "runs"
RUNS.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{RUNS / 'mlflow.db'}")
mlflow.set_experiment("tp1-softmax")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Device: {device}")

## 1. Data and features

Scikit-learn provides the official train and test sets. Removing headers, footers and quotes makes the classifier rely more on the body of each post. A stratified 80/20 split of the official training set keeps class proportions similar for training and validation.

NLTK tokenizes the text and handles optional stopword removal, stemming or lemmatization. Scikit-learn learns the TF-IDF vocabulary and inverse document frequencies from the training partition, then applies them to validation. TF-IDF gives more weight to terms that occur in a document and are less common across the corpus.

In [ ]:
for resource, location in {
    "stopwords": "corpora/stopwords", "wordnet": "corpora/wordnet.zip",
    "punkt": "tokenizers/punkt", "punkt_tab": "tokenizers/punkt_tab",
}.items():
    try:
        nltk.data.find(location)
    except LookupError:
        nltk.download(resource, quiet=True, raise_on_error=True)

dataset_args = {"remove": ("headers", "footers", "quotes")}
training = fetch_20newsgroups(subset="train", **dataset_args)
train_indices, val_indices = train_test_split(
    np.arange(len(training.data)), test_size=0.2, random_state=SEED, stratify=training.target
)
text_cache = {}

def processed_training(params):
    key = (params["remove_stopwords"], params["normalization"])
    if key not in text_cache:
        text_cache[key] = [preprocess_text(text, *key) for text in training.data]
    return text_cache[key]

def vectorizer_args(params):
    return {
        "ngram_range": (1, params["ngram_max"]),
        "max_features": params["max_features"],
        "min_df": params["min_df"],
    }

def validation_loaders(params):
    texts = processed_training(params)
    train_features, val_features, _ = vectorize_data(
        [texts[index] for index in train_indices],
        [texts[index] for index in val_indices],
        **vectorizer_args(params),
    )
    batch_size = params["batch_size"]
    train_labels = training.target[train_indices]
    train_loader = make_loader(train_features, train_labels, batch_size, shuffle=True, seed=SEED)
    val_loader = make_loader(val_features, training.target[val_indices], batch_size)
    train_eval = torch.utils.data.DataLoader(train_loader.dataset, batch_size=batch_size)
    return train_features.shape[1], train_loader, val_loader, train_eval

print(f"{len(train_indices)} training documents, {len(val_indices)} validation documents")

## 2. Model and hyperparameter search

The PyTorch model produces one linear score (logit) per topic. Cross-entropy applies log-softmax internally, and Adam updates the weights using the gradients calculated by backpropagation. With two logits, the probability of class 1 is $e^{z_1}/(e^{z_0}+e^{z_1})=\sigma(z_1-z_0)$. Because the difference of two linear logits is also linear in the features, this reduces to binary logistic regression.

Optuna proposes combinations of preprocessing and training settings and ranks the trials by validation accuracy after five epochs. Every trial uses the same split and seed. MLflow stores parameters and metrics so we can compare the runs.

In [ ]:
def build_model(input_dim, learning_rate):
    torch.manual_seed(SEED)
    model = SoftmaxRegression(input_dim, len(training.target_names)).to(device)
    return model, torch.optim.Adam(model.parameters(), lr=learning_rate)

def objective(trial):
    params = {
        "remove_stopwords": trial.suggest_categorical("remove_stopwords", [False, True]),
        "normalization": trial.suggest_categorical("normalization", ["none", "stem", "lemmatize"]),
        "ngram_max": trial.suggest_categorical("ngram_max", [1, 2]),
        "max_features": trial.suggest_categorical("max_features", [2000, 5000, 10000]),
        "min_df": trial.suggest_categorical("min_df", [1, 2, 5]),
        "lr": trial.suggest_float("lr", 1e-3, 1e-1, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }
    input_dim, train_loader, val_loader, _ = validation_loaders(params)
    model, optimizer = build_model(input_dim, params["lr"])
    history = train_classifier(model, train_loader, optimizer, 5, device, validation_loader=val_loader)
    with mlflow.start_run(run_name=f"search-trial-{trial.number}"):
        mlflow.log_params(params)
        for row in history:
            mlflow.log_metrics({"val_acc": row["val_acc"], "val_loss": row["val_loss"]}, step=row["epoch"])
        mlflow.log_metric("training_seconds", history[-1]["elapsed_s"])
    return history[-1]["val_acc"]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=15)
results = pd.DataFrame([
    {"trial": trial.number, "val_acc": trial.value, **trial.params} for trial in study.trials
]).sort_values("val_acc", ascending=False)
display(results.reset_index(drop=True).round(4))
best = study.best_params

## 3. Learning curves and convergence time

The three runs use the winning preprocessing configuration and the same split, batch size and seed. Only the learning rate changes. At the end of every epoch the model is evaluated on both splits. TensorBoard records the loss and accuracy curves for the timed runs.

Here, **time to convergence** means the first epoch at which validation accuracy reaches 0.72, a common target fixed for all three runs. A run that never reaches the target is marked as such. The table also shows the best observed validation epoch and total duration. Times include training and per-epoch scoring and exclude text processing, vectorization and experiment logging.

In [ ]:
TARGET_ACCURACY = 0.72
EXPERIMENT_EPOCHS = 15
input_dim, train_loader, val_loader, train_eval = validation_loaders(best)
rates = {"Slower LR": best["lr"] / 10, "Optuna LR": best["lr"], "Faster LR": best["lr"] * 3}
curves, summary = {}, []
for label, rate in rates.items():
    train_loader.generator.manual_seed(SEED)
    model, optimizer = build_model(input_dim, rate)
    history = train_classifier(
        model, train_loader, optimizer, EXPERIMENT_EPOCHS, device,
        validation_loader=val_loader, train_evaluation_loader=train_eval,
    )
    curve = pd.DataFrame(history)
    curves[label] = curve
    peak = curve.loc[curve.val_acc.idxmax()]
    reached = curve[curve.val_acc >= TARGET_ACCURACY]
    summary.append({
        "case": label, "lr": rate, "best_epoch": int(peak.epoch), "best_val_acc": peak.val_acc,
        "time_to_target_s": reached.elapsed_s.iloc[0] if len(reached) else None,
        "total_s": curve.elapsed_s.iloc[-1],
    })
    with SummaryWriter(RUNS / "tp1" / label) as writer:
        for row in history:
            for metric in ("train_acc", "val_acc", "train_loss", "val_loss"):
                writer.add_scalar(metric, row[metric], row["epoch"])

timing = pd.DataFrame(summary).set_index("case")
display(timing.round(4).fillna("Not reached"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for label, curve in curves.items():
    axes[0].plot(curve.epoch, curve.val_acc, label=label)
    axes[1].plot(curve.epoch, curve.val_loss, label=label)
    axes[2].plot(curve.epoch, curve.train_acc - curve.val_acc, label=label)
for axis, title in zip(axes, ("Validation accuracy", "Validation cross-entropy", "Train–validation accuracy gap")):
    axis.set(title=title, xlabel="Epoch")
axes[0].axhline(TARGET_ACCURACY, color="grey", linestyle=":")
axes[0].legend()
plt.tight_layout()
plt.show()

## 4. Final evaluation

Use the Optuna learning rate and pick the training duration from its validation curve. If several epochs tie, keep the earliest one. Fit TF-IDF again on the full official training set, train a fresh model for the selected duration, and score the official test set once.

In [ ]:
chosen_epochs = int(timing.loc["Optuna LR", "best_epoch"])
test = fetch_20newsgroups(subset="test", **dataset_args)
test_texts = [
    preprocess_text(text, best["remove_stopwords"], best["normalization"]) for text in test.data
]
train_features, test_features, _ = vectorize_data(
    processed_training(best), test_texts, **vectorizer_args(best)
)
train_loader = make_loader(train_features, training.target, best["batch_size"], shuffle=True, seed=SEED)
test_loader = make_loader(test_features, test.target, best["batch_size"])
model, optimizer = build_model(train_features.shape[1], best["lr"])
train_classifier(model, train_loader, optimizer, chosen_epochs, device)
test_metrics = evaluate_classifier(model, test_loader, device)
with mlflow.start_run(run_name="final"):
    mlflow.log_params({**best, "epochs": chosen_epochs})
    mlflow.log_metrics({"test_acc": test_metrics["accuracy"], "test_loss": test_metrics["loss"]})
print(f"Chosen epochs: {chosen_epochs} | Test accuracy: {test_metrics['accuracy']:.4f}")
print(classification_report(test_metrics["targets"], test_metrics["predictions"], target_names=test.target_names))
fig, axis = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(
    test_metrics["targets"], test_metrics["predictions"], display_labels=test.target_names,
    xticks_rotation="vertical", include_values=False, ax=axis,
)
plt.tight_layout()
plt.show()

## AI-use statement

I used AI assistance to reorganize the code into reusable modules, implement the timed experiments, and draft explanations and analysis. The notebook outputs record the experiments used to check the reported results.